# CRITERIO DE PRIORIZACIÓN PARA EVALUACIONES FASE II

## Cargue de capas "Llamados ciudadanos"

In [ ]:
import pandas as pd

df_ciudadanos = pd.read_excel("priorizacion_vulnerabilidad_riesgo_visitas.xlsx")
df_ciudadanos.head()

## Cargue capas catastrales

In [1]:
import geopandas as gpd
import requests

BASE = (
    "https://services8.arcgis.com/ljfiJpg35HWgdtaC/arcgis/rest/services/"
    "Validacion_geografica_WFL1/FeatureServer"
)

LAYERS = {
    "alfanumerica": 7,
    "urbano_construccion": 3,
    "urbano_terreno": 2,
    "unidad_construccion_urbano": 1,
}


def fetch_layer(layer_id: int) -> gpd.GeoDataFrame:
    """Download every feature of a layer as a GeoDataFrame (EPSG:4326)."""
    features = []
    offset = 0
    while True:
        resp = requests.get(
            f"{BASE}/{layer_id}/query",
            params={
                "where": "1=1",
                "outFields": "*",
                "outSR": "4326",
                "f": "geojson",
                "resultOffset": offset,
            },
            timeout=120,
        )
        resp.raise_for_status()
        payload = resp.json()
        if "error" in payload:
            raise RuntimeError(f"Layer {layer_id} query failed: {payload['error']}")
        page = payload.get("features", [])
        if not page:
            break
        features.extend(page)
        offset += len(page)
    return gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")


capas = {name: fetch_layer(lid) for name, lid in LAYERS.items()}
for name, gdf_ in capas.items():
    print(f"{name}: {len(gdf_)} registros, geometria={gdf_.geom_type.iloc[0] if len(gdf_) else 'N/A'}")

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))